<a href="https://colab.research.google.com/github/JaimeTS/hyrox-analysis/blob/main/notebooks/01_scraping_hyrox_limpio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Scraping de resultados HYROX

Objetivo: descargar los resultados de una prueba concreta desde HyResult y convertirlos en una base de datos estructurada para análisis de rendimiento, pacing y diferencias por edad y sexo.

ORDEN PARA DESCARGAR LOS DATOS.

Para todos hay que seguir el siguiente orden.

TODOS: Ejecutar el bloque completo

HOMBRES: 1-3 + 4

MUJERES: 1-3 + 5 (no el 4)

1. Librerías y configuración
2. Funciones generales
   2.1 extraer_ranking_pagina()
   2.2 descargar_categoria_completa()
   2.3 extraer_splits_atleta()
   2.4 descargar_splits_categoria()

3. Parámetros del evento

4. HYROX Men Barcelona 2026
   4.1 Ranking hombres
   4.2 Splits hombres

5. HYROX Women Barcelona 2026
   5.1 Ranking mujeres
   5.2 Splits mujeres

6. Unión Men + Women (opcional)

## 1. Librerías y configuración

In [1]:
import os
import math
import time
from io import StringIO

import requests
import pandas as pd
from bs4 import BeautifulSoup

BASE_URL = "https://www.hyresult.com"

headers = {
    "User-Agent": "Mozilla/5.0"
}

RUTA_DATOS = "/content/drive/MyDrive/Colab Notebooks/Hyrox"

os.makedirs(RUTA_DATOS, exist_ok=True)

## 2. Funciones generales

### 2.1 Función para extraer una página del ranking

In [2]:
def extraer_ranking_pagina(url):
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    rows = soup.find_all("tr")
    resultados = []

    i = 0

    while i < len(rows):
        cells = rows[i].find_all("td")
        textos = [c.get_text(" ", strip=True) for c in cells]

        # ==========================================================
        # CASO NORMAL
        # ==========================================================
        if len(cells) >= 7:

            posicion = cells[1].get_text(strip=True)
            posicion_ag = cells[2].get_text(strip=True)

            name_cell = cells[3]
            link_athlete = name_cell.find("a")
            img_flag = name_cell.find("img")

            nombre = link_athlete.get_text(strip=True) if link_athlete else None
            athlete_url = (
                link_athlete["href"]
                if link_athlete and link_athlete.has_attr("href")
                else None
            )

            pais = (
                img_flag["title"]
                if img_flag and img_flag.has_attr("title")
                else None
            )

            grupo_edad = cells[4].get_text(strip=True)
            tiempo = cells[5].get_text(strip=True)

            analyze_link = cells[6].find("a")
            analyze_url = (
                analyze_link["href"]
                if analyze_link and analyze_link.has_attr("href")
                else None
            )

            if nombre and tiempo and analyze_url:

                resultados.append({
                    "posicion": posicion,
                    "posicion_grupo_edad": posicion_ag,
                    "nombre": nombre,
                    "pais": pais,
                    "grupo_edad": grupo_edad,
                    "tiempo": tiempo,
                    "athlete_url": athlete_url,
                    "analyze_url": analyze_url
                })

            i += 1
            continue

        # ==========================================================
        # CASO ESPECIAL A
        # ==========================================================
        if len(cells) >= 3 and i + 4 < len(rows):

            posicion = cells[1].get_text(strip=True)
            posicion_ag = cells[2].get_text(strip=True)

            row_nombre = rows[i + 1]
            row_grupo = rows[i + 2]
            row_tiempo = rows[i + 3]
            row_analyze = rows[i + 4]

            nombre_cell = row_nombre.find("td")
            link_athlete = nombre_cell.find("a") if nombre_cell else None
            img_flag = nombre_cell.find("img") if nombre_cell else None

            nombre = link_athlete.get_text(strip=True) if link_athlete else None
            athlete_url = (
                link_athlete["href"]
                if link_athlete and link_athlete.has_attr("href")
                else None
            )

            pais = (
                img_flag["title"]
                if img_flag and img_flag.has_attr("title")
                else None
            )

            grupo_edad = row_grupo.get_text(" ", strip=True)
            tiempo = row_tiempo.get_text(" ", strip=True)

            analyze_link = row_analyze.find("a")
            analyze_url = (
                analyze_link["href"]
                if analyze_link and analyze_link.has_attr("href")
                else None
            )

            if nombre and tiempo and analyze_url:

                resultados.append({
                    "posicion": posicion,
                    "posicion_grupo_edad": posicion_ag,
                    "nombre": nombre,
                    "pais": pais,
                    "grupo_edad": grupo_edad,
                    "tiempo": tiempo,
                    "athlete_url": athlete_url,
                    "analyze_url": analyze_url
                })

                i += 5
                continue

        # ==========================================================
        # CASO ESPECIAL C
        # (fila vacía + datos repartidos)
        # ==========================================================
        if len(cells) == 1 and textos == [""] and i + 6 < len(rows):

            row_pos = rows[i + 1]
            row_pos_ag = rows[i + 2]
            row_nombre = rows[i + 3]
            row_grupo = rows[i + 4]
            row_tiempo = rows[i + 5]
            row_analyze = rows[i + 6]

            posicion = row_pos.get_text(" ", strip=True)
            posicion_ag = row_pos_ag.get_text(" ", strip=True)
            nombre = row_nombre.get_text(" ", strip=True)
            grupo_edad = row_grupo.get_text(" ", strip=True)
            tiempo = row_tiempo.get_text(" ", strip=True)

            analyze_link = row_analyze.find("a")
            analyze_url = (
                analyze_link["href"]
                if analyze_link and analyze_link.has_attr("href")
                else None
            )

            if (
                posicion.isdigit()
                and posicion_ag.isdigit()
                and nombre
                and tiempo
                and analyze_url
            ):

                resultados.append({
                    "posicion": posicion,
                    "posicion_grupo_edad": posicion_ag,
                    "nombre": nombre,
                    "pais": None,
                    "grupo_edad": grupo_edad,
                    "tiempo": tiempo,
                    "athlete_url": None,
                    "analyze_url": analyze_url
                })

                i += 7
                continue

        # ==========================================================
        # CASO ESPECIAL B
        # ==========================================================
        if len(cells) >= 2 and i + 5 < len(rows):

            posicion = cells[1].get_text(strip=True)

            row_pos_ag = rows[i + 1]
            row_nombre = rows[i + 2]
            row_grupo = rows[i + 3]
            row_tiempo = rows[i + 4]
            row_analyze = rows[i + 5]

            posicion_ag = row_pos_ag.get_text(" ", strip=True)

            nombre_cell = row_nombre.find("td")
            link_athlete = nombre_cell.find("a") if nombre_cell else None
            img_flag = nombre_cell.find("img") if nombre_cell else None

            nombre = link_athlete.get_text(strip=True) if link_athlete else None
            athlete_url = (
                link_athlete["href"]
                if link_athlete and link_athlete.has_attr("href")
                else None
            )

            pais = (
                img_flag["title"]
                if img_flag and img_flag.has_attr("title")
                else None
            )

            grupo_edad = row_grupo.get_text(" ", strip=True)
            tiempo = row_tiempo.get_text(" ", strip=True)

            analyze_link = row_analyze.find("a")
            analyze_url = (
                analyze_link["href"]
                if analyze_link and analyze_link.has_attr("href")
                else None
            )

            if nombre and tiempo and analyze_url:

                resultados.append({
                    "posicion": posicion,
                    "posicion_grupo_edad": posicion_ag,
                    "nombre": nombre,
                    "pais": pais,
                    "grupo_edad": grupo_edad,
                    "tiempo": tiempo,
                    "athlete_url": athlete_url,
                    "analyze_url": analyze_url
                })

                i += 6
                continue

        i += 1

    df = pd.DataFrame(resultados)

    if len(df) > 0:

        df["athlete_url"] = df["athlete_url"].apply(
            lambda x: BASE_URL + x if isinstance(x, str) and x.startswith("/") else x
        )

        df["analyze_url"] = df["analyze_url"].apply(
            lambda x: BASE_URL + x if isinstance(x, str) and x.startswith("/") else x
        )

    return df

### 2.2 Función para descargar una categoría completa

In [3]:
def descargar_categoria_completa(ranking_url, total_atletas, pausa=0.5):
    participantes_por_pagina = 100
    num_paginas = math.ceil(total_atletas / participantes_por_pagina)

    dfs = []

    for pagina in range(1, num_paginas + 1):
        url = f"{ranking_url}?p={pagina}"

        print(f"Descargando página {pagina}/{num_paginas}")

        df_pagina = extraer_ranking_pagina(url)
        dfs.append(df_pagina)

        time.sleep(pausa)

    df_categoria = pd.concat(dfs, ignore_index=True)

    return df_categoria

### 2.3 Función robusta para extraer splits

In [4]:
def extraer_splits_atleta(fila_atleta):
    url_resultado = fila_atleta["analyze_url"]
    url_splits = url_resultado + "?tab=splits"

    response = requests.get(url_splits, headers=headers)

    tablas = pd.read_html(StringIO(response.text))

    tablas_limpias = []

    for tabla in tablas:
        if tabla.shape[1] != 4:
            continue

        tabla = tabla.copy()
        tabla.columns = ["Split", "Time of Day", "Time", "Diff"]

        tabla = tabla.dropna(how="all")
        tabla = tabla[tabla["Split"].astype(str) != "Split"]

        tablas_limpias.append(tabla)

    if len(tablas_limpias) == 0:
        raise ValueError("No se encontraron tablas válidas de splits")

    df_splits = pd.concat(tablas_limpias, ignore_index=True)

    df_splits["posicion"] = fila_atleta["posicion"]
    df_splits["nombre"] = fila_atleta["nombre"]
    df_splits["grupo_edad"] = fila_atleta["grupo_edad"]
    df_splits["pais"] = fila_atleta["pais"]
    df_splits["tiempo_total"] = fila_atleta["tiempo"]
    df_splits["analyze_url"] = fila_atleta["analyze_url"]

    return df_splits

### 2.4 Función para descargar splits de una categoría

In [5]:
def descargar_splits_categoria(df_categoria, pausa=0.5):
    dfs_splits = []
    errores = []

    for i in range(len(df_categoria)):
        fila = df_categoria.iloc[i]

        try:
            print(f"Extrayendo {i+1}/{len(df_categoria)}: {fila['nombre']}")

            df_temp = extraer_splits_atleta(fila)
            dfs_splits.append(df_temp)

            time.sleep(pausa)

        except Exception as e:
            print(f"Error en {fila['nombre']}: {e}")

            errores.append({
                "indice": i,
                "posicion": fila["posicion"],
                "nombre": fila["nombre"],
                "analyze_url": fila["analyze_url"],
                "error": str(e)
            })

    df_splits = pd.concat(dfs_splits, ignore_index=True)
    df_errores = pd.DataFrame(errores)

    return df_splits, df_errores

## 3. Parámetros del evento

In [6]:
evento = "barcelona2026"

categorias = {
    "men": {
        "ranking_url": "https://www.hyresult.com/ranking/s8-2026-barcelona-hyrox-men",
        "total_atletas": 2350
    },
    "women": {
        "ranking_url": "https://www.hyresult.com/ranking/s8-2026-barcelona-hyrox-women",
        "total_atletas": 1296
    }
}

## 4. HYROX Men Barcelona 2026

### 4.1 Ranking hombres

In [7]:
ruta_ranking_men = f"{RUTA_DATOS}/barcelona2026_hyrox_men_ranking.csv"

if os.path.exists(ruta_ranking_men):
    print("Cargando ranking masculino desde Drive")
    df_men = pd.read_csv(ruta_ranking_men)
else:
    print("Descargando ranking masculino")
    df_men = descargar_categoria_completa(
        ranking_url=categorias["men"]["ranking_url"],
        total_atletas=categorias["men"]["total_atletas"],
        pausa=0.5
    )
    df_men.to_csv(ruta_ranking_men, index=False)

print(df_men.shape)
df_men.head()

Descargando ranking masculino
Descargando página 1/24
Descargando página 2/24
Descargando página 3/24
Descargando página 4/24
Descargando página 5/24
Descargando página 6/24
Descargando página 7/24
Descargando página 8/24
Descargando página 9/24
Descargando página 10/24
Descargando página 11/24
Descargando página 12/24
Descargando página 13/24
Descargando página 14/24
Descargando página 15/24
Descargando página 16/24
Descargando página 17/24
Descargando página 18/24
Descargando página 19/24
Descargando página 20/24
Descargando página 21/24
Descargando página 22/24
Descargando página 23/24
Descargando página 24/24
(2349, 8)


,posicion,posicion_grupo_edad,nombre,pais,grupo_edad,tiempo,athlete_url,analyze_url
0,1,1,Gaetan Gianni,France,16-24,53:44,https://www.hyresult.com/athlete/gaetan-gianni,https://www.hyresult.com/result/LR3MS4JI502C48
1,2,1,Emilio Aguayo,Spain,35-39,56:38,https://www.hyresult.com/athlete/emilio-aguayo,https://www.hyresult.com/result/LR3MS4JI501875
2,3,1,Loic Cebelieu,France,25-29,57:17,https://www.hyresult.com/athlete/loic-cebelieu,https://www.hyresult.com/result/LR3MS4JI5010E6
3,4,1,Ferran Bochaca Sabarich,Spain,30-34,57:24,https://www.hyresult.com/athlete/ferran-bochac...,https://www.hyresult.com/result/LR3MS4JI501D84
4,5,2,Alberto Casado Aroca,Spain,30-34,57:38,https://www.hyresult.com/athlete/alberto-casad...,https://www.hyresult.com/result/LR3MS4JI502972


### 4.2 Splits hombres

In [8]:
ruta_splits_men = f"{RUTA_DATOS}/barcelona2026_hyrox_men_splits.csv"
ruta_errores_men = f"{RUTA_DATOS}/barcelona2026_hyrox_men_errores.csv"

if os.path.exists(ruta_splits_men):
    print("Cargando splits masculinos desde Drive")
    df_splits_men = pd.read_csv(ruta_splits_men)

    if os.path.exists(ruta_errores_men):
        df_errores_men = pd.read_csv(ruta_errores_men)
    else:
        df_errores_men = pd.DataFrame()
else:
    print("Descargando splits masculinos")
    df_splits_men, df_errores_men = descargar_splits_categoria(
        df_men,
        pausa=0.5
    )

    df_splits_men.to_csv(ruta_splits_men, index=False)
    df_errores_men.to_csv(ruta_errores_men, index=False)

print(df_splits_men.shape)
print("Errores:", len(df_errores_men))

Descargando splits masculinos
Extrayendo 1/2349: Gaetan Gianni
Extrayendo 2/2349: Emilio Aguayo
Extrayendo 3/2349: Loic Cebelieu
Extrayendo 4/2349: Ferran Bochaca Sabarich
Extrayendo 5/2349: Alberto Casado Aroca
Extrayendo 6/2349: Sebastien Rajkowski
Extrayendo 7/2349: Mehdi Berrouigat
Extrayendo 8/2349: Pau Nacenta Merinas
Extrayendo 9/2349: Aaron Timmerman
Extrayendo 10/2349: Walter Paladina
Extrayendo 11/2349: Ethan Smith
Extrayendo 12/2349: Mohamed Yassin Kattar
Extrayendo 13/2349: Yoel Ferrer
Extrayendo 14/2349: Russell Christie
Extrayendo 15/2349: Martin Casse
Extrayendo 16/2349: James Francis
Extrayendo 17/2349: Thomas Jurado
Extrayendo 18/2349: Robert Stols
Extrayendo 19/2349: Sam Booth
Extrayendo 20/2349: David San Miguel López
Extrayendo 21/2349: Xabier De Velasco
Extrayendo 22/2349: Javier Puyalto Monaj
Extrayendo 23/2349: Ivan Llorens Catala
Extrayendo 24/2349: Sam Garcia
Extrayendo 25/2349: John Murtagh
Extrayendo 26/2349: Julien Caujolle
Extrayendo 27/2349: Luis Caja Nach

## 5. HYROX Women Barcelona 2026

### 5.1 Ranking mujeres

In [9]:
ruta_ranking_women = f"{RUTA_DATOS}/barcelona2026_hyrox_women_ranking.csv"

if os.path.exists(ruta_ranking_women):
    print("Cargando ranking femenino desde Drive")
    df_women = pd.read_csv(ruta_ranking_women)
else:
    print("Descargando ranking femenino")
    df_women = descargar_categoria_completa(
        ranking_url=categorias["women"]["ranking_url"],
        total_atletas=categorias["women"]["total_atletas"],
        pausa=0.5
    )
    df_women.to_csv(ruta_ranking_women, index=False)

print(df_women.shape)
df_women.head()

Descargando ranking femenino
Descargando página 1/13
Descargando página 2/13
Descargando página 3/13
Descargando página 4/13
Descargando página 5/13
Descargando página 6/13
Descargando página 7/13
Descargando página 8/13
Descargando página 9/13
Descargando página 10/13
Descargando página 11/13
Descargando página 12/13
Descargando página 13/13
(1296, 8)


,posicion,posicion_grupo_edad,nombre,pais,grupo_edad,tiempo,athlete_url,analyze_url
0,1,1,Marta Izquierdo Ayuso,Spain,25-29,1:03:59,https://www.hyresult.com/athlete/marta-izquier...,https://www.hyresult.com/result/LR3MS4JI502F6D
1,2,1,Julia Ernst,Germany,30-34,1:05:03,https://www.hyresult.com/athlete/julia-ernst,https://www.hyresult.com/result/LR3MS4JI5026B5
2,3,2,Michela Borzoni,United Kingdom,30-34,1:05:57,https://www.hyresult.com/athlete/michela-borzoni,https://www.hyresult.com/result/LR3MS4JI503921
3,4,3,Lisa Gillman,Ireland,30-34,1:06:23,https://www.hyresult.com/athlete/lisa-gillman,https://www.hyresult.com/result/LR3MS4JI501390
4,5,1,Cristina Villarreal,Mexico,16-24,1:07:03,https://www.hyresult.com/athlete/cristina-vill...,https://www.hyresult.com/result/LR3MS4JI5010C1


### 5.2 Splits Mujeres

In [10]:
ruta_splits_women = f"{RUTA_DATOS}/barcelona2026_hyrox_women_splits.csv"
ruta_errores_women = f"{RUTA_DATOS}/barcelona2026_hyrox_women_errores.csv"

if os.path.exists(ruta_splits_women):
    print("Cargando splits femeninos desde Drive")
    df_splits_women = pd.read_csv(ruta_splits_women)

    if os.path.exists(ruta_errores_women):
        df_errores_women = pd.read_csv(ruta_errores_women)
    else:
        df_errores_women = pd.DataFrame()
else:
    print("Descargando splits femeninos")
    df_splits_women, df_errores_women = descargar_splits_categoria(
        df_women,
        pausa=0.5
    )

    df_splits_women.to_csv(ruta_splits_women, index=False)
    df_errores_women.to_csv(ruta_errores_women, index=False)

print(df_splits_women.shape)
print("Errores:", len(df_errores_women))

Descargando splits femeninos
Extrayendo 1/1296: Marta Izquierdo Ayuso
Extrayendo 2/1296: Julia Ernst
Extrayendo 3/1296: Michela Borzoni
Extrayendo 4/1296: Lisa Gillman
Extrayendo 5/1296: Cristina Villarreal
Extrayendo 6/1296: Erin Ireland
Extrayendo 7/1296: Yessica Pérez Torrente
Extrayendo 8/1296: Laetitia Duhamel
Extrayendo 9/1296: Laura Hawkins
Extrayendo 10/1296: Anna Kirsch
Extrayendo 11/1296: Gracious Phillips
Extrayendo 12/1296: Malen Barceló
Extrayendo 13/1296: Claudia Grau
Extrayendo 14/1296: Valeria Andreini
Extrayendo 15/1296: Maria Fernanda Lopez
Extrayendo 16/1296: Eva Den Uijl
Extrayendo 17/1296: Sophia Öjhammar
Extrayendo 18/1296: Lisa Liberio
Extrayendo 19/1296: ANNA SANTIDRIAN
Extrayendo 20/1296: Britt Gillott
Extrayendo 21/1296: Linda De Wildt
Extrayendo 22/1296: María del Camino del Blanco
Extrayendo 23/1296: Joséphine Segard
Extrayendo 24/1296: Sydnie Schindler
Extrayendo 25/1296: Laura Salvia
Extrayendo 26/1296: Sami Stoute
Extrayendo 27/1296: Catherine Schroeder
E

## 6. Unión Men + Women

In [11]:
df_men["sexo"] = "men"
df_women["sexo"] = "women"

df_splits_men["sexo"] = "men"
df_splits_women["sexo"] = "women"

df_ranking_total = pd.concat(
    [df_men, df_women],
    ignore_index=True
)

df_splits_total = pd.concat(
    [df_splits_men, df_splits_women],
    ignore_index=True
)

df_ranking_total.to_csv(
    f"{RUTA_DATOS}/{evento}_hyrox_ranking_total.csv",
    index=False
)

df_splits_total.to_csv(
    f"{RUTA_DATOS}/{evento}_hyrox_splits_total.csv",
    index=False
)

print(df_ranking_total.shape)
print(df_splits_total.shape)

(3645, 9)
(109300, 11)
